# Ejercicio 01 · Dataset público y SVM reproducible
**Pregunta:** con la información disponible al terminar el 1.er semestre, ¿puede un SVM identificar si un estudiante abandonará, seguirá matriculado o se graduará?

- **Dataset:** *Predict Students' Dropout and Academic Success* (UCI, id 697, CC BY 4.0, Instituto Politécnico de Portalegre).
- **Unidad de análisis:** un estudiante matriculado. **Target:** `Target` (Dropout, Enrolled, Graduate).
- **Métrica principal:** F1 macro; recall de `Dropout` como criterio de desempate.
- **Advertencia:** es un ejercicio académico. No sirve para decidir sobre estudiantes reales sin supervisión humana y evaluación de equidad.

In [ ]:
import sys, platform, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd, sklearn, matplotlib
import matplotlib.pyplot as plt
from inf8239_u01.data import load_dataset, DEFAULT_PATH, EXPECTED_SHAPE, UCI_URL, download_csv
from inf8239_u01.features import TARGET, DROP_COLUMNS, split_features_target
from inf8239_u01.models import build_dropout_baseline, build_dropout_svm
from inf8239_u01.evaluation import summarize, per_class_table, error_table

REPORTS = ROOT / "reports"; REPORTS.mkdir(exist_ok=True)
DATA = ROOT / DEFAULT_PATH
SEED = 42
env = f"Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | scikit-learn {sklearn.__version__} | matplotlib {matplotlib.__version__} | {platform.platform()}"
(REPORTS / "environment.txt").write_text(env + "\n", encoding="utf-8")
print(env)

## 1. Descarga reproducible y carga

In [ ]:
if not DATA.exists():
    download_csv(UCI_URL, destination=DATA, sep=";")
df = load_dataset(DATA)
print(df.shape, "esperado:", EXPECTED_SHAPE)
assert df.shape == EXPECTED_SHAPE
df.head(3)

## 2. Auditoría de datos

In [ ]:
audit = pd.DataFrame({"tipo": df.dtypes.astype(str), "ausentes": df.isna().sum(),
                      "unicos": df.nunique()})
print("Duplicados:", df.duplicated().sum(), "| ausentes totales:", int(df.isna().sum().sum()))
dist = df[TARGET].value_counts()
print(pd.DataFrame({"n": dist, "porcentaje": (dist / len(df) * 100).round(1)}))
audit.to_csv(REPORTS / "auditoria_columnas.csv")
audit.head(8)

La base no tiene ausentes ni duplicados. El problema está en el **desbalance**: `Enrolled` es la clase minoritaria (≈18 %) y un clasificador que siempre responde `Graduate` acierta casi la mitad de los casos.

## 3. Diccionario y prevención de fuga

In [ ]:
dic = pd.read_csv(ROOT / "docs" / "diccionario_datos.csv")
print("Diccionario:", dic.shape)
print("En el CSV y no en el diccionario:", set(df.columns) - set(dic["variable"]))
print("En el diccionario y no en el CSV:", set(dic["variable"]) - set(df.columns))
fuga = dic[dic["variable"].isin(DROP_COLUMNS)][["variable", "momento_disponibilidad", "riesgo"]]
fuga.to_csv(REPORTS / "columnas_excluidas.csv", index=False)
pd.set_option("display.max_colwidth", 90)
fuga

**Decisión.** El momento de predicción es el **fin del 1.er semestre**. Se excluyen las seis variables del 2.º semestre (no existen en ese momento) y `Debtor` y `Tuition fees up to date` (pueden registrarse cuando el estudiante ya está en riesgo de abandonar). Se conservan las del 1.er semestre, que sí están disponibles al cierre. Los atributos sensibles (estado civil, nacionalidad, sexo, estudios y ocupación de los padres) siguen en el modelo; se declara el riesgo de equidad en la conclusión.

In [ ]:
X, y = split_features_target(df)
assert TARGET not in X.columns and not set(DROP_COLUMNS) & set(X.columns)
print("Variables predictoras:", X.shape[1], "| excluidas:", len(DROP_COLUMNS))

## 4. Partición
80 % entrenamiento y 20 % prueba, estratificada, `random_state=42`. El escalado y la codificación están **dentro del Pipeline**, así que se ajustan solo con entrenamiento. El conjunto de prueba se usa una sola vez, al final.

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.20, random_state=SEED, stratify=y)
print(Xtr.shape, Xte.shape)
print(pd.DataFrame({"train": ytr.value_counts(normalize=True).round(3),
                    "test": yte.value_counts(normalize=True).round(3)}))

## 5. Baseline y SVM

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
models = {
    "Baseline (clase mayoritaria)": build_dropout_baseline(X),
    "SVM RBF (C=1)": build_dropout_svm(X),
    "SVM RBF balanceado (C=1)": build_dropout_svm(X, class_weight="balanced"),
}
rows = []
for name, model in models.items():
    cv_scores = cross_val_score(model, Xtr, ytr, cv=cv, scoring="f1_macro")
    model.fit(Xtr, ytr)
    rows.append({"modelo": name, "f1_macro_cv_media": cv_scores.mean(),
                 "f1_macro_cv_std": cv_scores.std(),
                 **{f"{k}_prueba": v for k, v in summarize(yte, model.predict(Xte)).items()}})
results = pd.DataFrame(rows).set_index("modelo").round(4)
results

## 6. Ajuste de hiperparámetros (solo con validación cruzada sobre entrenamiento)

In [ ]:
grid = GridSearchCV(
    build_dropout_svm(X, class_weight="balanced"),
    param_grid={"model__C": [0.1, 1, 10], "model__gamma": ["scale", 0.001, 0.01]},
    scoring="f1_macro", cv=cv, n_jobs=-1, return_train_score=True)
grid.fit(Xtr, ytr)
cv_table = pd.DataFrame(grid.cv_results_)[["param_model__C", "param_model__gamma",
            "mean_test_score", "std_test_score", "mean_train_score"]].sort_values(
            "mean_test_score", ascending=False).round(4)
cv_table.to_csv(REPORTS / "svm_grid_cv.csv", index=False)
print("Mejor:", grid.best_params_, "| F1 macro CV:", round(grid.best_score_, 4))
cv_table

## 7. Evaluación final en prueba

In [ ]:
final = grid.best_estimator_
pred = final.predict(Xte)
final_row = {"modelo": f"SVM balanceado ajustado {grid.best_params_}",
             "f1_macro_cv_media": grid.best_score_, "f1_macro_cv_std": cv_table.iloc[0]["std_test_score"],
             **{f"{k}_prueba": v for k, v in summarize(yte, pred).items()}}
metrics = pd.concat([results.reset_index(), pd.DataFrame([final_row])]).set_index("modelo").round(4)
metrics.to_csv(REPORTS / "metrics.csv")
per_class = per_class_table(yte, pred).round(3)
per_class.to_csv(REPORTS / "per_class_final.csv")
base_class = per_class_table(yte, models["SVM RBF (C=1)"].predict(Xte)).round(3)
base_class.to_csv(REPORTS / "per_class_svm_base.csv")
print(metrics.to_string())
print()
print(per_class)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
labels = ["Dropout", "Enrolled", "Graduate"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for ax, (title, p) in zip(axes, [("SVM base", models["SVM RBF (C=1)"].predict(Xte)),
                                 ("SVM balanceado (final)", pred)]):
    ConfusionMatrixDisplay.from_predictions(yte, p, labels=labels, ax=ax, cmap="Blues",
                                            colorbar=False)
    ax.set_title(title); ax.set_xlabel("Clase predicha"); ax.set_ylabel("Clase real")
plt.tight_layout(); plt.savefig(REPORTS / "confusion_matrices.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(3); w = 0.38
ax.bar(x - w/2, base_class.loc[labels, "f1-score"], w, label="SVM base", color="#94a3b8")
ax.bar(x + w/2, per_class.loc[labels, "f1-score"], w, label="SVM balanceado", color="#0f7c7b")
ax.set_xticks(x, labels); ax.set_ylabel("F1 por clase"); ax.set_ylim(0, 1)
ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig(REPORTS / "f1_por_clase.png", dpi=150); plt.show()

## 8. Análisis de errores

In [ ]:
errors = error_table(Xte, yte, pred)
errors.to_csv(REPORTS / "error_analysis_full.csv", index=False)
conf = pd.crosstab(yte, pd.Series(pred, index=yte.index, name="predicho"))
rate = (1 - np.diag(conf.values) / conf.sum(axis=1).values).round(3)
print(f"Errores: {len(errors)} de {len(yte)} ({len(errors)/len(yte):.1%})")
print(pd.Series(rate, index=conf.index, name="tasa_de_error_por_clase"))
conf

In [ ]:
pairs = (errors.groupby(["real", "predicho"]).size().sort_values(ascending=False)
         .rename("casos").reset_index())
pairs["porcentaje_de_errores"] = (pairs["casos"] / len(errors) * 100).round(1)
pairs.to_csv(REPORTS / "confusiones_principales.csv", index=False)
cols = ["real", "predicho", "Course", "Age at enrollment", "Admission grade",
        "Curricular units 1st sem (approved)", "Curricular units 1st sem (grade)"]
sample = errors[cols].groupby(["real", "predicho"], group_keys=False).head(2).head(8)
sample.to_csv(REPORTS / "error_analysis.csv", index=False)
print(pairs.to_string(index=False)); sample

In [ ]:
# ¿Qué distingue a los errores? Comparación de medias en variables del 1.er semestre
comp = pd.DataFrame({
    "aciertos": Xte[(yte.values == pred)][["Curricular units 1st sem (approved)",
                                              "Curricular units 1st sem (grade)", "Admission grade"]].mean(),
    "errores": errors[["Curricular units 1st sem (approved)",
                       "Curricular units 1st sem (grade)", "Admission grade"]].mean()}).round(2)
comp.to_csv(REPORTS / "aciertos_vs_errores.csv")
comp

## 9. Conclusión

Con la información disponible al cierre del 1.er semestre, un SVM sí identifica el abandono mucho mejor que un baseline, pero no lo hace con la fiabilidad que exigiría una decisión sobre personas. El baseline que siempre responde `Graduate` obtuvo F1 macro 0.222 y exactitud 0.499. El SVM con preprocesamiento llegó a 0.598 en prueba, y su versión con `class_weight="balanced"` a 0.640 (validación cruzada 0.648 ± 0.015). Que prueba y validación cruzada coincidan indica que la cifra no depende de una partición afortunada. El ajuste de `C` y `gamma` no mejoró el resultado: la combinación ganadora (`C=1`, `gamma="scale"`) es la configuración por defecto, y `C=10` con `gamma="scale"` sobreajusta (0.97 en entrenamiento frente a 0.63 en validación).

La mejora de 4.2 puntos de F1 macro tiene un costo que el promedio oculta. El modelo balanceado recupera la clase `Enrolled` (F1 de 0.26 a 0.46), pero detecta menos abandonos: el recall de `Dropout` baja de 0.676 a 0.560, es decir, 33 de los 284 estudiantes que abandonaron dejan de ser detectados. Como el error más costoso es no detectar un abandono, el criterio de desempate cuestiona la elección que dicta la métrica principal. Mantengo el modelo balanceado porque el protocolo fija F1 macro como métrica principal, pero declaro que, si la institución prioriza no perder abandonos, el SVM sin balanceo es la opción defendible.

El análisis de errores muestra dónde se pierde información. De 885 estudiantes de prueba, 277 se clasifican mal (31.3 %). La confusión más frecuente es `Dropout` predicho como `Enrolled` (83 casos, 30 % de los errores), seguida de `Graduate` predicho como `Enrolled` (66). `Enrolled` es una clase ambigua por naturaleza: describe a quien aún no terminó, no a un perfil distinto. Además, los errores casi no difieren de los aciertos en las variables del primer semestre (unidades aprobadas 4.52 frente a 4.82), por lo que otras variables no registradas probablemente explican parte del abandono.

Hay limitaciones que no resolví. La evaluación usa una sola partición de prueba de 885 casos, por lo que diferencias de uno o dos puntos no son concluyentes. El modelo conserva atributos sensibles (estado civil, nacionalidad, sexo, estudios y ocupación de los padres) y no medí si sus errores se reparten por igual entre grupos. Los datos provienen de un solo instituto portugués y no justifican extrapolar a otro contexto.

Antes de usar este enfoque en otro dominio, debo repetir la auditoría con ese dominio: confirmar en qué momento existe cada variable para evitar fuga, fijar de antemano cuál error es más costoso y revisar la equidad por grupo. Y en cualquier uso real, la predicción debe ser una alerta para una persona que acompañe al estudiante, no una decisión automática.